# 基础分配器换手惩罚固定对照
只将额外换手惩罚0.006改为0，保留10bp训练成本。3个固定种子，复用控制模型，新拟合3个模型；两组三种Tiered费用独立原生账户，不改变现役SIM。已复用历史不作为独立样本外。

In [1]:
import os,json,qlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/allocator_turnover_ablation_20261014'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
from src.workflow.allocator_turnover_ablation import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
summary

backtest loop:   0%|          | 0/3775 [00:00<?, ?it/s]

backtest loop:   0%|          | 0/3775 [00:00<?, ?it/s]

backtest loop:   0%|          | 0/3775 [00:00<?, ?it/s]

backtest loop:   0%|          | 0/3775 [00:00<?, ?it/s]

backtest loop:   0%|          | 0/3775 [00:00<?, ?it/s]

backtest loop:   0%|          | 0/3775 [00:00<?, ?it/s]

{'experiment': 'allocator_turnover_ablation_20261014',
 'plan_sha256': 'c581e857ea79369c9bd2002161ca47400f3b206ed6197e0725dec46c3612e3c5',
 'decision': 'not_promoted_fixed_turnover_ablation_failed',
 'gates': {'net_soxx_excess_all_costs': False,
  'drawdown_30pct': True,
  'cvar_no_worse': True,
  'beat_paired_control_all_costs': False,
  'rolling_70pct': False,
  'annual_70pct': False,
  'bootstrap_positive': False,
  'independent_development': False,
  'qualified_forward': False},
 'objective_achieved': False,
 'new_fits': 3,
 'new_native_cases': 6,
 'reused_control_models': 3,
 'reused_native_cases': 6,
 'recorder_id': '9137bb22b8dc4d6d92688decb91db2e5',
 'experiment_id': '948010085854765276',
 'recorder_uri': '/Users/rziiiii/dev/workspace/research/allocator_turnover_ablation_20261014/qlib_recorder',
 'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/allocator_turnover_ablation_20261014/qlib_recorder/948010085854765276/9137bb22b8dc4d6d92688decb91db2e5',
 'provider_uri': '

# 基础网络换手惩罚固定对照：失败

只将额外换手惩罚从0.006改为0，保留10bp训练交易成本及网络、输入、300训练轮次、最终epoch检查点、共享辅助损失、权重约束、EMA/源持仓过滤及完整Stage1控制器。3个固定种子71/172/273，0次控制重拟合、3次候选拟合；两种ensemble各运行0/2/5bp，共6个新Qlib原生账户，另复用6个原策略/SOXX账户。

初始化哈希、训练行数/首末日期/成熟时点、辅助样本数、scaler和训练轮次均与各自保存基线配对核验。控制模型重新推理与已保存预测一致，控制ensemble平滑目标也与来源逐行一致。模型按训练前信息截止约束过滤标签；没有用外窗选择seed或epoch。

控制是已冻结的三seed重训base ensemble，**不是f696旧单模型**，因此旧策略单列。完整账户保持10万美元、当前IBKR Pro Tiered费表投影、09:46决策/09:47成交以及原约束。源虚拟持仓平滑口径沿用配对父证据，两组相同；实际账户按自己的股数、现金、月度费用及风险状态独立结算。

|方案|滑点bp|累计净收益%|监测最大回撤%|平均期末现金%|订单数|
|---|---:|---:|---:|---:|---:|
|control|0|280.24|-26.20|46.25|1286|
|control|2|269.16|-25.77|46.43|1284|
|control|5|266.08|-25.83|46.89|1258|
|candidate|0|82.71|-26.77|64.03|1410|
|candidate|2|76.49|-28.89|65.81|1347|
|candidate|5|67.29|-25.69|65.60|1351|
|current_anchor|0|228.99|-29.50|48.19|1281|
|soxx|0|387.75|-43.38|0.15|1|
|current_anchor|2|279.67|-27.07|48.31|1248|
|soxx|2|387.73|-43.39|0.14|1|
|current_anchor|5|41.12|-31.42|58.30|1006|
|soxx|5|387.70|-43.39|0.12|1|

候选三个费用情景都大幅落后配对控制，也均未超过SOXX。2bp候选/控制相对终值0.4781，并非改善；回撤与CVaR门通过，但年度、滚动和区块置信下界等收益门失败。2bp候选2023–2026年度相对SOXX财富分别约0.8293/0.8737/0.6529/0.7649，四年均落后。

## 现金方向与原假设相反

|种子|控制原始现金均值%|去惩罚候选原始现金均值%|
|---:|---:|---:|
|71|40.95|56.28|
|172|39.96|61.24|
|273|33.93|54.33|

三种子都在进入风控之前就增加现金，不能将候选现金上升全部归因于后续风控。完整账户2bp现金约65.81%，高于控制46.43%；候选订单也更多。这反驳“惩罚抑制参与，删除即可回收现金”的简单机制。不能凭本实验断言0.006是全局最优，亦不能断言所有正则化都有益；但本固定零惩罚路线没有继续推广的证据。

决定：**not_promoted_fixed_turnover_ablation_failed**。固定路线封存，不搜索中间系数、不换最佳seed、不放宽风险门。稳定净SOXX超额目标未完成，生产SIM未改。

风险：2023+历史已经复用，缺独立开发与前向资格；这里的完整账户结果是研究回测，非实际模拟盘成交。监测回撤来自原生日收益及已知/收盘净值点，不是全分钟最大回撤认证。当前费用投影及复权/真实分红现金流边界继承来源。

下一信息需求：保留有效正则化，检验基础分配器的现金信号在未来收益/尾部风险上的排序和跨年度稳定性，先区分“学到的防守信息不足”和“信息到目标的映射失效”，而不是继续调整本轮惩罚系数。已失败的月度更新、风控重新武装和此项零惩罚机制均不作为下一轮参数搜索起点。

[直接基准比较](backtest_benchmark_comparison.csv)、[年度](annual_metrics.csv)、[区块区间](uncertainty.csv)、[种子现金](seed_cash.csv)、[Recorder清单](verification_manifest.json)保留全部结果。6项相关检查通过，模型与大账户对象仅在NAS Recorder。
